In [12]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [13]:
df_org=pd.read_csv('Telco_churn_dataset.csv')

In [14]:
df_org.sample(4)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
136,7850-VWJUU,Female,0,No,No,23,Yes,No,Fiber optic,Yes,...,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),75.00,1778.5,No
1851,2485-ITVKB,Female,0,Yes,No,2,No,No phone service,DSL,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,35.10,68.75,Yes
790,7131-ZQZNK,Female,0,Yes,Yes,60,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),59.85,3590.2,No
6441,5084-OOVCJ,Female,0,Yes,Yes,17,Yes,No,DSL,Yes,...,No,Yes,No,No,Month-to-month,Yes,Credit card (automatic),55.35,920.5,No


In [15]:
X = df_org.drop('Churn', axis=1)
Y = df_org['Churn']

In [16]:
Y = Y.map({
    'Yes':1,
    'No':0
})
Y = Y.astype(int)

In [17]:
Y.head()

0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int32

In [18]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (FunctionTransformer)
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# Evaluation Metrics
from sklearn.metrics import (

    accuracy_score,

    classification_report,

    confusion_matrix
)




num_columns = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges'
]

multi_columns = [
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod'
]

binary_columns = [
    'gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'PaperlessBilling',
]


def basic_cleaning(df1):

    df1 = df1.copy()

    # Remove customerID
    df1 = df1.drop('customerID', axis=1)

    # Convert TotalCharges
    df1['TotalCharges'] = pd.to_numeric(df1['TotalCharges'],errors='coerce')

    # Fill missing values
    df1['TotalCharges'] = df1['TotalCharges'].fillna(df1['TotalCharges'].median())

    # Binary mapping
    binary_map = {
        'Yes':1,
        'No':0,
        'Female':0,
        'Male':1
    }

    for col in binary_columns:
        df1[col] = df1[col].map(binary_map)

    return df1

In [20]:
preprocessor = ColumnTransformer([('num', StandardScaler(), num_columns ),
                                  ('cat', OneHotEncoder(drop='first'),multi_columns)
                                  ],remainder='passthrough')

In [31]:
svm_base = Pipeline([
    ('cleaning',FunctionTransformer(basic_cleaning)),
    ('preprocessing',preprocessor),
    ('model', SVC())])

# Train Model
svm_pipeline.fit(

    X_train,

    y_train
)

# Prediction
baseline_pred = svm_pipeline.predict(

    X_test
)

# Accuracy
baseline_acc = accuracy_score(

    y_test,

    baseline_pred
)

print(
    "Baseline Accuracy : ",

    baseline_acc
)

Baseline Accuracy :  0.8140525195173882


In [32]:
from sklearn.model_selection import GridSearchCV


# Create Pipeline
svm_pipeline = Pipeline([
    ('cleaning',FunctionTransformer(basic_cleaning)),
    ('preprocessing',preprocessor),
    ('model', SVC())])

# Parameter Grid
param_grid = {

    'model__C': [

        0.1,

        1,

        10,

        100
    ],

    'model__kernel': [

        'linear',

        'rbf',

        'poly'
    ],

    'model__gamma': [

        'scale',

        'auto'
    ]
}

# GridSearchCV
grid_svm = GridSearchCV(

    estimator=svm_pipeline,

    param_grid=param_grid,

    cv=5,

    scoring='accuracy',

    verbose=1
)

# Train Model
grid_svm.fit(

    X_train,

    y_train
)
     

Fitting 5 folds for each of 24 candidates, totalling 120 fits


,estimator,"Pipeline(step...del', SVC())])"
,param_grid,"{'model__C': [0.1, 1, ...], 'model__gamma': ['scale', 'auto'], 'model__kernel': ['linear', 'rbf', ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,func,<function bas...0021D573E9300>


In [33]:

print("Best Parameters : ")

print(grid_svm.best_params_)

print("\nBest Accuracy : ")

print(grid_svm.best_score_)

Best Parameters : 
{'model__C': 1, 'model__gamma': 'scale', 'model__kernel': 'poly'}

Best Accuracy : 
0.799608984067795


In [34]:
svm_results = pd.DataFrame(

    grid_svm.cv_results_
)

svm_results[[

    'param_model__C',

    'param_model__kernel',

    'param_model__gamma',

    'mean_test_score'
]]

,param_model__C,param_model__kernel,param_model__gamma,mean_test_score
0,0.1,linear,scale,0.797125
1,0.1,rbf,scale,0.792333
2,0.1,poly,scale,0.791445
3,0.1,linear,auto,0.797125
4,0.1,rbf,auto,0.793575
5,0.1,poly,auto,0.734469
6,1.0,linear,scale,0.795883
7,1.0,rbf,scale,0.797302
8,1.0,poly,scale,0.799609
9,1.0,linear,auto,0.795883


In [26]:
best_svm_model = grid_svm.best_estimator_

best_pred = best_svm_model.predict(

    X_test
)

best_acc = accuracy_score(

    y_test,

    best_pred
)

print(
    "Tuned Model Accuracy : ",

    best_acc
)

Tuned Model Accuracy :  0.8097941802696949


In [27]:

comparison_df = pd.DataFrame({

    'Model': ['Baseline SVM','Tuned SVM'],

    'Accuracy': [baseline_acc,best_acc]
})

comparison_df
     

,Model,Accuracy
0,Baseline SVM,0.814053
1,Tuned SVM,0.809794


In [36]:
svm_base.fit(X_train, y_train)

,steps,"[('cleaning', ...), ('preprocessing', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,func,<function bas...0021D573E9300>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None


In [37]:
svm_pipeline.fit(X_train, y_train)

,steps,"[('cleaning', ...), ('preprocessing', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,func,<function bas...0021D573E9300>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None


In [39]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Baseline model predictions
baseline_pred = svm_base.predict(X_test)
baseline_prob = svm_base.predict_proba(X_test)[:,1]

# Tuned model predictions
tuned_pred =svm_pipeline.predict(X_test)
tuned_prob = svm_pipeline.predict_proba(X_test)[:,1]

# Comparison dataframe
comparison_df = pd.DataFrame({

    'Model': ['Baseline SVM', 'Tuned SVM'],

    'Accuracy': [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, tuned_pred)
    ],

    'Precision': [
        precision_score(y_test, baseline_pred),
        precision_score(y_test, tuned_pred)
    ],

    'Recall': [
        recall_score(y_test, baseline_pred),
        recall_score(y_test, tuned_pred)
    ],

    'F1-Score': [
        f1_score(y_test, baseline_pred),
        f1_score(y_test, tuned_pred)
    ],

    'ROC-AUC': [
        roc_auc_score(y_test, baseline_prob),
        roc_auc_score(y_test, tuned_prob)
    ]
})

comparison_df

AttributeError: This 'Pipeline' has no attribute 'predict_proba'